<a href="https://colab.research.google.com/github/manaskanugo97-max/Healthcare-Lead-Gen-System/blob/main/website_analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/healthcare_lead_gen_project

/content/drive/MyDrive/healthcare_lead_gen_project


In [ ]:
import os

print("Project files:")
print(os.listdir())

print("\nData files:")
print(os.listdir("data"))

Project files:
['data', 'config.py', '__pycache__', 'osm_scraper.py', 'online_presence_checker.py']

Data files:
['healthcare_osm_raw.csv', 'healthcare_online_presence.csv']


In [ ]:
%%writefile config.py

import os

PROJECT_DIR = "/content/drive/MyDrive/healthcare_lead_gen_project"
DATA_DIR = os.path.join(PROJECT_DIR, "data")

# Step 1 output
RAW_OSM_CSV = os.path.join(DATA_DIR, "healthcare_osm_raw.csv")

# Step 2 output
ONLINE_PRESENCE_CSV = os.path.join(DATA_DIR, "healthcare_online_presence.csv")

# Step 3 output
WEBSITE_ANALYSIS_CSV = os.path.join(DATA_DIR, "healthcare_website_analysis.csv")

# Final output
FINAL_CSV = os.path.join(DATA_DIR, "healthcare_leads_final.csv")

Overwriting config.py


In [ ]:
from config import ONLINE_PRESENCE_CSV, WEBSITE_ANALYSIS_CSV
import os

print("Step 3 input file:")
print(ONLINE_PRESENCE_CSV)

print("\nDoes Step 3 input file exist?")
print(os.path.exists(ONLINE_PRESENCE_CSV))

print("\nStep 3 output file will be:")
print(WEBSITE_ANALYSIS_CSV)

Step 3 input file:
/content/drive/MyDrive/healthcare_lead_gen_project/data/healthcare_online_presence.csv

Does Step 3 input file exist?
True

Step 3 output file will be:
/content/drive/MyDrive/healthcare_lead_gen_project/data/healthcare_website_analysis.csv


In [ ]:
!pip install requests beautifulsoup4 pandas -q

In [ ]:
%%writefile website_analyzer.py

import os
import re
import time
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

from config import ONLINE_PRESENCE_CSV, WEBSITE_ANALYSIS_CSV


HEADERS = {
    "User-Agent": "Mozilla/5.0 HealthcareLeadGenBot/1.0"
}


def clean_value(value):
    if pd.isna(value):
        return ""
    return str(value).strip()


def normalize_url(url):
    """
    Adds https:// if missing.
    """
    url = clean_value(url)

    if not url:
        return ""

    if not url.startswith("http://") and not url.startswith("https://"):
        url = "https://" + url

    return url


def fetch_website(url):
    """
    Try to open website safely.
    """
    url = normalize_url(url)

    if not url:
        return None, "", 0, "No website URL available"

    try:
        response = requests.get(
            url,
            headers=HEADERS,
            timeout=10,
            allow_redirects=True
        )

        return response, response.text, response.status_code, "Website opened"

    except Exception as e:
        return None, "", 0, str(e)


def has_keyword(text, keywords):
    text = text.lower()

    for keyword in keywords:
        if keyword.lower() in text:
            return True

    return False


def detect_contact_page(soup, base_url):
    """
    Checks if website has contact page link.
    """
    if soup is None:
        return False

    contact_keywords = ["contact", "contact us", "appointment", "book appointment"]

    for link in soup.find_all("a", href=True):
        link_text = link.get_text(" ", strip=True).lower()
        href = link["href"].lower()

        if any(word in link_text for word in contact_keywords):
            return True

        if any(word in href for word in ["contact", "appointment"]):
            return True

    return False


def detect_services_page(soup):
    """
    Checks if website has service/treatment page.
    """
    if soup is None:
        return False

    service_keywords = [
        "services",
        "treatments",
        "departments",
        "specialities",
        "specialties",
        "doctors",
        "clinic",
        "hospital"
    ]

    for link in soup.find_all("a", href=True):
        link_text = link.get_text(" ", strip=True).lower()
        href = link["href"].lower()

        if any(word in link_text for word in service_keywords):
            return True

        if any(word in href for word in service_keywords):
            return True

    return False


def detect_phone_or_email(text):
    """
    Checks whether phone number or email appears on website homepage.
    """
    email_pattern = r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
    phone_pattern = r"(\+91[\-\s]?)?[6-9]\d{9}"

    email_found = re.search(email_pattern, text) is not None
    phone_found = re.search(phone_pattern, text) is not None

    return phone_found, email_found


def analyze_single_website(website_url):
    """
    Classify one website as:
    No Website / Poor Website / Good Website
    """

    website_url = clean_value(website_url)

    if not website_url:
        return {
            "Website Classification": "No Website",
            "Website Status Code": 0,
            "HTTPS Used": "No",
            "Contact Page Found": "No",
            "Services Page Found": "No",
            "Website Analysis Reason": "No official website was found."
        }

    response, html, status_code, fetch_message = fetch_website(website_url)

    if response is None or status_code == 0:
        return {
            "Website Classification": "Poor Website",
            "Website Status Code": status_code,
            "HTTPS Used": "Yes" if website_url.startswith("https://") else "No",
            "Contact Page Found": "No",
            "Services Page Found": "No",
            "Website Analysis Reason": f"Website URL exists but could not be opened properly. Reason: {fetch_message}"
        }

    final_url = response.url
    https_used = "Yes" if final_url.startswith("https://") else "No"

    if status_code >= 400:
        return {
            "Website Classification": "Poor Website",
            "Website Status Code": status_code,
            "HTTPS Used": https_used,
            "Contact Page Found": "No",
            "Services Page Found": "No",
            "Website Analysis Reason": f"Website returned error status code {status_code}."
        }

    soup = BeautifulSoup(html, "html.parser")
    page_text = soup.get_text(" ", strip=True).lower()

    contact_page_found = detect_contact_page(soup, final_url)
    services_page_found = detect_services_page(soup)
    phone_found, email_found = detect_phone_or_email(page_text)

    score = 0
    reasons = []

    if https_used == "Yes":
        score += 20
        reasons.append("uses HTTPS")
    else:
        reasons.append("does not clearly use HTTPS")

    if contact_page_found:
        score += 25
        reasons.append("has contact/appointment page")
    else:
        reasons.append("contact/appointment page not clearly found")

    if services_page_found:
        score += 25
        reasons.append("has services/treatment related page")
    else:
        reasons.append("services/treatment page not clearly found")

    if phone_found:
        score += 15
        reasons.append("phone number visible on website")

    if email_found:
        score += 15
        reasons.append("email address visible on website")

    if len(page_text) > 800:
        score += 10
        reasons.append("homepage has reasonable content")
    else:
        reasons.append("homepage has limited content")

    if score >= 60:
        classification = "Good Website"
    else:
        classification = "Poor Website"

    return {
        "Website Classification": classification,
        "Website Status Code": status_code,
        "HTTPS Used": https_used,
        "Contact Page Found": "Yes" if contact_page_found else "No",
        "Services Page Found": "Yes" if services_page_found else "No",
        "Website Analysis Reason": "; ".join(reasons)
    }


def analyze_websites(
    input_file=ONLINE_PRESENCE_CSV,
    output_file=WEBSITE_ANALYSIS_CSV,
    limit=None
):
    """
    Reads Step 2 file and performs website classification.
    """

    print("Step 3 started")
    print("Input file:", input_file)
    print("Output file:", output_file)

    if not os.path.exists(input_file):
        raise FileNotFoundError(
            f"Input file not found: {input_file}. Please complete Step 2 first."
        )

    df = pd.read_csv(input_file)

    if limit is not None:
        df = df.head(limit)

    analysis_results = []

    print("Total records to analyze:", len(df))

    for index, row in df.iterrows():
        business_name = row.get("Business Name", "")
        official_website = row.get("Official Website Found", "")

        print(f"[{index + 1}/{len(df)}] Analyzing website for: {business_name}")

        result = analyze_single_website(official_website)
        analysis_results.append(result)

        time.sleep(1)

    analysis_df = pd.DataFrame(analysis_results)

    final_df = pd.concat(
        [df.reset_index(drop=True), analysis_df.reset_index(drop=True)],
        axis=1
    )

    final_df.to_csv(output_file, index=False, encoding="utf-8-sig")

    print("Step 3 completed")
    print("Saved file:", output_file)

    return final_df


if __name__ == "__main__":
    df = analyze_websites(
        input_file=ONLINE_PRESENCE_CSV,
        output_file=WEBSITE_ANALYSIS_CSV,
        limit=None
    )

    print(df.head())

Overwriting website_analyzer.py


In [ ]:
import pandas as pd
from config import WEBSITE_ANALYSIS_CSV

df3 = pd.read_csv(WEBSITE_ANALYSIS_CSV)

def repair_website_classification(row):
    website = str(row.get("Official Website Found", "")).strip()
    old_website = str(row.get("Website URL", "")).strip()
    status_code = row.get("Website Status Code", 0)
    https_used = str(row.get("HTTPS Used", "")).strip()
    contact_page = str(row.get("Contact Page Found", "")).strip()
    services_page = str(row.get("Services Page Found", "")).strip()

    if not website and not old_website:
        return "No Website"

    try:
        status_code = int(status_code)
    except:
        status_code = 0

    if status_code == 0 or status_code >= 400:
        return "Poor Website"

    if https_used == "Yes" and contact_page == "Yes" and services_page == "Yes":
        return "Good Website"

    return "Poor Website"


df3["Website Classification"] = df3.apply(repair_website_classification, axis=1)

df3.to_csv(WEBSITE_ANALYSIS_CSV, index=False, encoding="utf-8-sig")

print("Website Classification column added/fixed.")
print(df3[["Business Name", "Website Classification", "Website Analysis Reason"]].head())

Website Classification column added/fixed.


KeyError: "['Website Analysis Reason'] not in index"

In [ ]:
!python website_analyzer.py

Step 3 started
Input file: /content/drive/MyDrive/healthcare_lead_gen_project/data/healthcare_online_presence.csv
Output file: /content/drive/MyDrive/healthcare_lead_gen_project/data/healthcare_website_analysis.csv
Total records to analyze: 100
[1/100] Analyzing website for: Greater Kailash Hospital
[2/100] Analyzing website for: Qurban Husain
[3/100] Analyzing website for: Shri Aurobindo dental clinic and institute of medical science
[4/100] Analyzing website for: Dr. Mutha
[5/100] Analyzing website for: Arshid shah
[6/100] Analyzing website for: Shubham Hospital
[7/100] Analyzing website for: Aditya Nursing Home
[8/100] Analyzing website for: Curewell hospital janjeerwala square
[9/100] Analyzing website for: kjd
[10/100] Analyzing website for: Family Dental Clinic
[11/100] Analyzing website for: Yoga Amrutam
[12/100] Analyzing website for: Suhan Pathology
[13/100] Analyzing website for: Dr. Lal Path Labs
[14/100] Analyzing website for: Dr. Vrinda Vashistha
[15/100] Analyzing website

In [ ]:
import pandas as pd
from config import WEBSITE_ANALYSIS_CSV

df3 = pd.read_csv(WEBSITE_ANALYSIS_CSV)

print("Step 3 CSV file path:")
print(WEBSITE_ANALYSIS_CSV)

print("\nTotal rows:")
print(len(df3))

print("\nAll columns in Step 3 CSV:")
print(df3.columns.tolist())

print("\nColumn check:")
print("Website Classification exists:", "Website Classification" in df3.columns)
print("Website Analysis Reason exists:", "Website Analysis Reason" in df3.columns)

Step 3 CSV file path:
/content/drive/MyDrive/healthcare_lead_gen_project/data/healthcare_website_analysis.csv

Total rows:
100

All columns in Step 3 CSV:
['Unnamed: 0', 'Business Name', 'Industry Category', 'Business Description', 'Location', 'Latitude', 'Longitude', 'Google Maps Profile Link', 'OpenStreetMap Link', 'Website URL', 'Phone Number', 'Email Address', 'Source', 'Official Website Found', 'Online Presence Type', 'Referral Platform Links', 'Search Result Links', 'Website Classification']

Column check:
Website Classification exists: True
Website Analysis Reason exists: False


In [ ]:
from config import WEBSITE_ANALYSIS_CSV
import os
import pandas as pd

print("Step 3 file exists:", os.path.exists(WEBSITE_ANALYSIS_CSV))
print("Step 3 file path:", WEBSITE_ANALYSIS_CSV)

df3 = pd.read_csv(WEBSITE_ANALYSIS_CSV)
print("Total rows:", len(df3))
df3.head()

Step 3 file exists: True
Step 3 file path: /content/drive/MyDrive/healthcare_lead_gen_project/data/healthcare_website_analysis.csv
Total rows: 100


,Unnamed: 0,Business Name,Industry Category,Business Description,Location,Latitude,Longitude,Google Maps Profile Link,OpenStreetMap Link,Website URL,Phone Number,Email Address,Source,Official Website Found,Online Presence Type,Referral Platform Links,Search Result Links,Website Classification
0,0,Greater Kailash Hospital,hospital,Hospital listed on OpenStreetMap,Indore,22.725082,75.890465,https://www.google.com/maps/search/?api=1&quer...,https://www.openstreetmap.org/node/1802056893,NaN,NaN,NaN,OpenStreetMap,https://www.gkh.co.in/,Official Website Found,https://www.medindia.net/directories/hospitals...,https://www.gkh.co.in/ | https://www.hexahealt...,Poor Website
1,1,Qurban Husain,doctors,Doctors listed on OpenStreetMap,Indore,22.716741,75.863790,https://www.google.com/maps/search/?api=1&quer...,https://www.openstreetmap.org/node/3360065915,NaN,NaN,NaN,OpenStreetMap,https://indushospital.org.pk/,Official Website Found,https://en.wikipedia.org/wiki/Indus_Hospital_a...,https://en.wikipedia.org/wiki/Indus_Hospital_a...,Poor Website
2,2,Shri Aurobindo dental clinic and institute of ...,clinic,Clinic listed on OpenStreetMap,Ujjain Road,22.791523,75.845636,https://www.google.com/maps/search/?api=1&quer...,https://www.openstreetmap.org/node/4273145396,NaN,NaN,NaN,OpenStreetMap,https://hospital.sriaurobindouniversity.edu.in/,Official Website Found,https://www.medindia.net/directories/hospitals...,https://hospital.sriaurobindouniversity.edu.in...,Poor Website
3,3,Dr. Mutha,doctors,Doctors listed on OpenStreetMap,Gita Bhavan intersection,22.718593,75.885588,https://www.google.com/maps/search/?api=1&quer...,https://www.openstreetmap.org/node/4310485189,NaN,+9198-26-150111,NaN,OpenStreetMap,https://gitabhawanhospital.org/,Official Website Found,https://www.justdial.com/Indore/Dr-Mutha-Sudar...,https://gitabhawanhospital.org/ | https://www....,Poor Website
4,4,Arshid shah,pharmacy,Pharmacy listed on OpenStreetMap,"Dhar road, 452001",22.709624,75.827079,https://www.google.com/maps/search/?api=1&quer...,https://www.openstreetmap.org/node/4522507595,NaN,8103012446,shah.arshid05@rediffmail.com,OpenStreetMap,https://www.latlong.net/poi/arshid-shah-411182,Official Website Found,NaN,https://www.latlong.net/poi/arshid-shah-411182...,Poor Website


In [ ]:
df = pd.read_csv(ONLINE_PRESENCE_CSV)

In [ ]:
df.to_csv(WEBSITE_ANALYSIS_CSV)